# 第3章: scikit-learn による分類器ツアー

この Notebook は、原本 `machine-learning-book/ch03/ch03.ipynb` を最新の scikit-learn / NumPy / Matplotlib 環境で継続検証できる形に移行したものです。
原本の「分類器を横断して比較する」という意図を保ちながら、Notebook マジック、古い API、ファイル出力セルを整理し、CI で安定して通る構成にしています。


## この Notebook で確認すること

- 現在の `uv` 環境で Python と主要パッケージのバージョンを確認する。
- 原本サブモジュールの図版を読み取り専用で再利用する。
- Iris データセットを使って、Perceptron、Logistic Regression、SVM、Decision Tree、Random Forest、k-NN を比較する。
- カーネル SVM 用の非線形データ例も軽量に再現する。
- `pytest --nbmake` のヘッドレス実行で完走することを確認する。


In [ ]:
from importlib.metadata import version
from pathlib import Path
import platform
import sys

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np
import pandas as pd
from IPython.display import Image, display
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, Perceptron, SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree

PACKAGE_NAMES = ['numpy', 'pandas', 'matplotlib', 'scikit-learn']

def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / '.github/workflows/ci.yml').exists() and (candidate / 'machine-learning-book').exists():
            return candidate
    raise FileNotFoundError('リポジトリルートを特定できませんでした')

REPO_ROOT = find_repo_root(Path.cwd())
FIG_DIR = REPO_ROOT / 'machine-learning-book' / 'ch03' / 'figures'
assert FIG_DIR.exists(), f'図版ディレクトリが見つかりません: {FIG_DIR}'

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(1)

print(f'Python 実行ファイル: {sys.executable}')
print(f'Python バージョン: {platform.python_version()}')
print(f'Matplotlib バックエンド: {matplotlib.get_backend()}')
print(f'原本 ch03 図版ディレクトリ: {FIG_DIR}')


In [ ]:
package_versions = pd.DataFrame(
    [(name, version(name)) for name in PACKAGE_NAMES],
    columns=['パッケージ', 'バージョン'],
)
package_versions


## 原本図版の参照

書籍の説明で使われる図版アセットは `machine-learning-book/ch03/figures/` に残し、移行版 Notebook からは読み取り専用で参照します。
実装コードやデータ処理はルート側に移しつつ、章の導入図は原本のまま確認できるようにしています。


In [ ]:
selected_figures = [
    ('03_01.png', 640),
    ('03_03.png', 500),
    ('03_09.png', 640),
    ('03_17.png', 500),
    ('03_23.png', 420),
]

for name, width in selected_figures:
    figure_path = FIG_DIR / name
    print(name)
    display(Image(filename=str(figure_path), width=width))


## Iris データの準備

原本どおり `petal length` と `petal width` の 2 特徴量を使い、70/30 の stratified split を作成します。
線形モデルや k-NN では標準化後の特徴量を使い、木系モデルでは元スケールをそのまま使います。


In [ ]:
iris = load_iris(as_frame=True)
X = iris.data[['petal length (cm)', 'petal width (cm)']].to_numpy()
y = iris.target.to_numpy()
class_names = iris.target_names.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=1,
    stratify=y,
)

scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_test_std = scaler.transform(X_test)

X_combined = np.vstack((X_train, X_test))
X_combined_std = np.vstack((X_train_std, X_test_std))
y_combined = np.hstack((y_train, y_test))
test_idx = np.arange(len(y_train), len(y_combined))

pd.DataFrame(
    {
        'split': ['full', 'train', 'test'],
        'counts': [np.bincount(y).tolist(), np.bincount(y_train).tolist(), np.bincount(y_test).tolist()],
    }
)


In [ ]:
def plot_decision_regions(
    X: np.ndarray,
    y: np.ndarray,
    classifier,
    test_idx=None,
    resolution: float = 0.02,
    label_names=None,
):
    markers = ('o', 's', '^', 'v', '<')
    colors = ('tab:red', 'tab:blue', 'tab:green', 'gray', 'cyan')
    cmap = ListedColormap(colors[: len(np.unique(y))])

    x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    x2_min, x2_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx1, xx2 = np.meshgrid(
        np.arange(x1_min, x1_max, resolution),
        np.arange(x2_min, x2_max, resolution),
    )
    grid = np.c_[xx1.ravel(), xx2.ravel()]
    lab = classifier.predict(grid).reshape(xx1.shape)

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.contourf(xx1, xx2, lab, alpha=0.25, cmap=cmap)
    ax.set_xlim(xx1.min(), xx1.max())
    ax.set_ylim(xx2.min(), xx2.max())

    for idx, cl in enumerate(np.unique(y)):
        if label_names is None:
            label = f'Class {cl}'
        else:
            label = label_names[cl]
        ax.scatter(
            x=X[y == cl, 0],
            y=X[y == cl, 1],
            alpha=0.85,
            c=colors[idx],
            marker=markers[idx],
            edgecolor='black',
            label=label,
        )

    if test_idx is not None and len(test_idx) > 0:
        X_test_highlight = X[test_idx, :]
        ax.scatter(
            X_test_highlight[:, 0],
            X_test_highlight[:, 1],
            c='none',
            edgecolor='black',
            linewidth=1.2,
            marker='o',
            s=90,
            label='test set',
        )
    return fig, ax


## Perceptron による最初の分類

章の冒頭どおり、まずは scikit-learn の `Perceptron` で Iris 3 クラス分類を試します。
評価値と決定領域を確認して、線形分類器の基礎的な振る舞いを押さえます。


In [ ]:
ppn = Perceptron(eta0=0.1, random_state=1)
ppn.fit(X_train_std, y_train)
y_pred = ppn.predict(X_test_std)

print(f'誤分類数: {(y_test != y_pred).sum()}')
print(f'accuracy_score: {accuracy_score(y_test, y_pred):.3f}')
print(f'model.score: {ppn.score(X_test_std, y_test):.3f}')

fig, ax = plot_decision_regions(X_combined_std, y_combined, classifier=ppn, test_idx=test_idx, label_names=class_names)
ax.set_xlabel('Petal length [standardized]')
ax.set_ylabel('Petal width [standardized]')
ax.set_title('Perceptron の決定領域')
ax.legend(loc='upper left')
fig.tight_layout()
plt.show()
plt.close(fig)


## Logistic Regression

原本と同様に、シグモイド関数とロジスティック損失の形を確認した上で、scikit-learn の `LogisticRegression` を使って多クラス分類を行います。
正則化係数 `C` を変えたときの重み変化も可視化します。


In [ ]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-np.clip(z, -250, 250)))

def loss_1(z: np.ndarray) -> np.ndarray:
    return -np.log(sigmoid(z))

def loss_0(z: np.ndarray) -> np.ndarray:
    return -np.log(1.0 - sigmoid(z))

z = np.arange(-10, 10, 0.1)
sigma_z = sigmoid(z)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(z, sigma_z)
axes[0].axvline(0.0, color='black', linewidth=1)
axes[0].set_xlabel('z')
axes[0].set_ylabel('sigmoid(z)')
axes[0].set_title('シグモイド関数')
axes[0].set_yticks([0.0, 0.5, 1.0])

axes[1].plot(sigma_z, loss_1(z), label='y=1')
axes[1].plot(sigma_z, loss_0(z), linestyle='--', label='y=0')
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 5.1)
axes[1].set_xlabel('sigmoid(z)')
axes[1].set_ylabel('loss')
axes[1].set_title('ロジスティック損失')
axes[1].legend(loc='best')

fig.tight_layout()
plt.show()
plt.close(fig)


In [ ]:
lr = LogisticRegression(C=100.0, solver='lbfgs', max_iter=200, random_state=1)
lr.fit(X_train_std, y_train)

proba_preview = pd.DataFrame(
    lr.predict_proba(X_test_std[:3, :]),
    columns=class_names,
)
proba_preview.index = [f'sample_{idx}' for idx in range(3)]
proba_preview


In [ ]:
fig, ax = plot_decision_regions(X_combined_std, y_combined, classifier=lr, test_idx=test_idx, label_names=class_names)
ax.set_xlabel('Petal length [standardized]')
ax.set_ylabel('Petal width [standardized]')
ax.set_title('Logistic Regression の決定領域')
ax.legend(loc='upper left')
fig.tight_layout()
plt.show()
plt.close(fig)

weights, params = [], []
for c in np.arange(-3, 4):
    model = LogisticRegression(C=10.0 ** c, solver='lbfgs', max_iter=200, random_state=1)
    model.fit(X_train_std, y_train)
    weights.append(model.coef_[1])
    params.append(10.0 ** c)

weights = np.array(weights)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(params, weights[:, 0], marker='o', label='Petal length')
ax.plot(params, weights[:, 1], marker='s', linestyle='--', label='Petal width')
ax.set_xscale('log')
ax.set_xlabel('C')
ax.set_ylabel('Weight coefficient for class 1')
ax.set_title('正則化による重みの変化')
ax.legend(loc='upper left')
plt.show()
plt.close(fig)


## 線形 SVM と SGDClassifier

線形 SVM で同じ Iris 問題を解き、Perceptron や Logistic Regression と決定領域を見比べます。
併せて、scikit-learn が提供する `SGDClassifier` の損失関数指定も簡単に確認します。


In [ ]:
svm_linear = SVC(kernel='linear', C=1.0, random_state=1)
svm_linear.fit(X_train_std, y_train)

fig, ax = plot_decision_regions(X_combined_std, y_combined, classifier=svm_linear, test_idx=test_idx, label_names=class_names)
ax.set_xlabel('Petal length [standardized]')
ax.set_ylabel('Petal width [standardized]')
ax.set_title('Linear SVM の決定領域')
ax.legend(loc='upper left')
fig.tight_layout()
plt.show()
plt.close(fig)

sgd_variants = pd.DataFrame(
    {
        'estimator': ['perceptron', 'log_loss', 'hinge'],
        'example_class': [
            SGDClassifier(loss='perceptron', random_state=1).__class__.__name__,
            SGDClassifier(loss='log_loss', random_state=1).__class__.__name__,
            SGDClassifier(loss='hinge', random_state=1).__class__.__name__,
        ],
    }
)
sgd_variants


## カーネル SVM と非線形分離

原本と同様に XOR 風の合成データを作り、RBF カーネルで非線形な境界が引けることを確かめます。
その後、Iris に対する RBF SVM の `gamma` の違いも比較します。


In [ ]:
X_xor = np.random.randn(200, 2)
y_xor = np.logical_xor(X_xor[:, 0] > 0, X_xor[:, 1] > 0)
y_xor = np.where(y_xor, 1, 0)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(X_xor[y_xor == 1, 0], X_xor[y_xor == 1, 1], c='royalblue', marker='s', label='Class 1')
ax.scatter(X_xor[y_xor == 0, 0], X_xor[y_xor == 0, 1], c='tomato', marker='o', label='Class 0')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.set_title('XOR 風データ')
ax.legend(loc='best')
plt.show()
plt.close(fig)

svm_rbf_xor = SVC(kernel='rbf', gamma=0.10, C=10.0, random_state=1)
svm_rbf_xor.fit(X_xor, y_xor)
fig, ax = plot_decision_regions(X_xor, y_xor, classifier=svm_rbf_xor, label_names=['Class 0', 'Class 1'])
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.set_title('RBF SVM on XOR-like data')
ax.legend(loc='upper left')
plt.show()
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for axis, gamma in zip(axes, [0.2, 100.0]):
    model = SVC(kernel='rbf', gamma=gamma, C=1.0, random_state=1)
    model.fit(X_train_std, y_train)
    x1_min, x1_max = X_combined_std[:, 0].min() - 1, X_combined_std[:, 0].max() + 1
    x2_min, x2_max = X_combined_std[:, 1].min() - 1, X_combined_std[:, 1].max() + 1
    xx1, xx2 = np.meshgrid(np.arange(x1_min, x1_max, 0.02), np.arange(x2_min, x2_max, 0.02))
    grid = np.c_[xx1.ravel(), xx2.ravel()]
    lab = model.predict(grid).reshape(xx1.shape)
    axis.contourf(xx1, xx2, lab, alpha=0.25, cmap=ListedColormap(['tab:red', 'tab:blue', 'tab:green']))
    for idx, cl in enumerate(np.unique(y_combined)):
        axis.scatter(X_combined_std[y_combined == cl, 0], X_combined_std[y_combined == cl, 1],
                     c=['tab:red', 'tab:blue', 'tab:green'][idx], marker=['o', 's', '^'][idx],
                     edgecolor='black', alpha=0.8)
    axis.set_title(f'gamma={gamma}')
    axis.set_xlabel('Petal length [standardized]')
    axis.set_ylabel('Petal width [standardized]')
fig.suptitle('Iris に対する RBF SVM の gamma 比較')
fig.tight_layout()
plt.show()
plt.close(fig)


## Decision Tree、Random Forest、k-NN

後半では、木ベースのモデルと近傍法を比較します。
不純度指標の形を先に確認し、その後に Decision Tree、Random Forest、k-NN の決定領域を描画します。


In [ ]:
def entropy(p: np.ndarray) -> np.ndarray:
    p = np.asarray(p, dtype=float)
    result = np.zeros_like(p)
    mask = (p > 0) & (p < 1)
    result[mask] = -p[mask] * np.log2(p[mask]) - (1 - p[mask]) * np.log2(1 - p[mask])
    return result

def gini(p: np.ndarray) -> np.ndarray:
    return 2 * p * (1 - p)

def classification_error(p: np.ndarray) -> np.ndarray:
    return 1 - np.maximum(p, 1 - p)

x = np.arange(0.0, 1.0, 0.01)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x, entropy(x), label='Entropy', linewidth=2)
ax.plot(x, 0.5 * entropy(x), label='Entropy (scaled)', linewidth=2)
ax.plot(x, gini(x), linestyle='--', label='Gini impurity', linewidth=2)
ax.plot(x, classification_error(x), linestyle='-.', label='Misclassification error', linewidth=2)
ax.set_xlabel('p(i=1)')
ax.set_ylabel('Impurity index')
ax.set_ylim(0, 1.1)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.18), ncol=2)
plt.show()
plt.close(fig)


In [ ]:
tree_model = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=1)
tree_model.fit(X_train, y_train)

fig, ax = plot_decision_regions(X_combined, y_combined, classifier=tree_model, test_idx=test_idx, label_names=class_names)
ax.set_xlabel('Petal length [cm]')
ax.set_ylabel('Petal width [cm]')
ax.set_title('Decision Tree の決定領域')
ax.legend(loc='upper left')
fig.tight_layout()
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 6))
plot_tree(
    tree_model,
    feature_names=['Petal length', 'Petal width'],
    class_names=class_names,
    filled=True,
    ax=ax,
)
ax.set_title('Decision Tree の構造')
plt.show()
plt.close(fig)

forest = RandomForestClassifier(n_estimators=25, random_state=1, n_jobs=1)
forest.fit(X_train, y_train)
fig, ax = plot_decision_regions(X_combined, y_combined, classifier=forest, test_idx=test_idx, label_names=class_names)
ax.set_xlabel('Petal length [cm]')
ax.set_ylabel('Petal width [cm]')
ax.set_title('Random Forest の決定領域')
ax.legend(loc='upper left')
fig.tight_layout()
plt.show()
plt.close(fig)

knn = KNeighborsClassifier(n_neighbors=5, p=2, metric='minkowski')
knn.fit(X_train_std, y_train)
fig, ax = plot_decision_regions(X_combined_std, y_combined, classifier=knn, test_idx=test_idx, label_names=class_names)
ax.set_xlabel('Petal length [standardized]')
ax.set_ylabel('Petal width [standardized]')
ax.set_title('k-NN の決定領域')
ax.legend(loc='upper left')
fig.tight_layout()
plt.show()
plt.close(fig)

comparison = pd.DataFrame(
    [
        ('Perceptron', ppn.score(X_test_std, y_test)),
        ('LogisticRegression', lr.score(X_test_std, y_test)),
        ('Linear SVM', svm_linear.score(X_test_std, y_test)),
        ('DecisionTree', tree_model.score(X_test, y_test)),
        ('RandomForest', forest.score(X_test, y_test)),
        ('k-NN', knn.score(X_test_std, y_test)),
    ],
    columns=['model', 'test_accuracy'],
).sort_values('test_accuracy', ascending=False)
comparison


## まとめ

第3章の移行版では、原本の分類器ツアーを以下の形で継続検証可能にしました。

- Iris データセットは `scikit-learn` 同梱データを使い、外部ネットワークやローカル CSV に依存しないようにした。
- Perceptron、Logistic Regression、線形 SVM、RBF SVM、Decision Tree、Random Forest、k-NN を現在の API で再現した。
- `SGDClassifier(loss='log')` のような古い指定は使わず、現行の `log_loss` などに合わせた。
- 原本の図版は読み取り専用サブモジュールから再利用し、Notebook 本体は `src/` 側に配置した。
